In [ ]:
%%spark

import os
import re
import sys
import traceback
from typing import Dict, Optional
from pyspark.sql import DataFrame, SparkSession
from datetime import datetime
from pyspark.sql.functions import col, md5


class ErroPipeline(Exception):
    def __init__(
        self,
        codigo: str,
        mensagem: str,
        etapa: Optional[str] = None,
        objeto: Optional[str] = None,
        detalhes: Optional[dict] = None,
        acao: Optional[str] = None,
    ) -> None:
        super().__init__(mensagem)
        self.codigo = str(codigo).strip() or "ERRO_PIPELINE"
        self.mensagem = str(mensagem).strip()
        self.etapa = str(etapa).strip() if etapa else None
        self.objeto = str(objeto).strip() if objeto else None
        self.detalhes = dict(detalhes or {})
        self.acao = str(acao).strip() if acao else None

    def __str__(self) -> str:
        partes = [f"[{self.codigo}] {self.mensagem}"]
        if self.objeto:
            partes.append(f"objeto={self.objeto}")
        return " | ".join(partes)


class ErroAcessoDados(ErroPipeline):
    pass


class ErroContratoDados(ErroPipeline):
    pass


class ErroOperacional(ErroPipeline):
    pass


def _causa_raiz(exc):
    atual = exc
    visitados = set()

    while atual is not None and id(atual) not in visitados:
        visitados.add(id(atual))
        proxima = atual.__cause__ or atual.__context__
        if proxima is None:
            return atual
        atual = proxima

    return atual or exc


def _redigir_segredos(valor) -> str:
    texto = str(valor)
    texto = re.sub(
        r"(?i)\b(password|senha|secret|token|keytab)\s*([=:])\s*([^\s,;]+)",
        r"\1\2***",
        texto,
    )
    texto = re.sub(
        r"(?i)([a-z][a-z0-9+.-]*://)([^/\s:@]+):([^@/\s]+)@",
        r"\1***:***@",
        texto,
    )
    return texto


def _texto_log_resumido(valor, limite: int = 1000) -> str:
    texto = " ".join(_redigir_segredos(valor).split())
    if len(texto) <= limite:
        return texto
    return texto[:limite] + "..."


def _detalhes_log_seguros(detalhes: dict) -> str:
    palavras_sensiveis = ("PASSWORD", "SENHA", "SECRET", "TOKEN", "KEYTAB")
    partes = []

    for chave in sorted((detalhes or {}).keys(), key=str):
        chave_texto = str(chave)
        if any(palavra in chave_texto.upper() for palavra in palavras_sensiveis):
            valor = "***"
        else:
            valor = _texto_log_resumido(detalhes[chave])
        partes.append(f"{chave_texto}={valor}")

    return "; ".join(partes)

def publicar_tabelas_ando(
    df: DataFrame,
    database: str,
    tabela: str,
    modo: str,
    coluna_origem: str = "CD_CLI",
) -> None:

    try:
        if not database or not tabela:
            print("[ERRO] Database ou tabela não informados. Publicação não realizada.")
            return

        tbl_ando = f"{database}.ANDO_{tabela}"
        col_ando = f"DA_{coluna_origem}"

        if coluna_origem not in df.columns:
            print(
                f"[ALERTA] Coluna origem '{coluna_origem}' não existe. "
                f"Tabela '{tbl_ando}' NÃO será publicada."
            )
            return

        try:
            df_tmp = (
                df.withColumn(
                    col_ando,
                    md5(col(coluna_origem).cast("string"))
                )
                .drop(coluna_origem)
            )

            colunas_finais = [
                col_ando if c == coluna_origem else c
                for c in df.columns
            ]

            df_ando = df_tmp.select(*colunas_finais)

        except Exception as e:
            print(
                f"[ERRO] Falha ao gerar coluna '{col_ando}'. "
                f"Tabela '{tbl_ando}' NÃO publicada. Erro: {e}"
            )
            return

        try:
            (
                df_ando.write
                .mode(modo)
                .insertInto(tbl_ando)
            )

            print(
                f"[OK] Publicação concluída: '{tbl_ando}' "
                f"com '{col_ando}' (CHAR(32)) e sem exposição de "
                f"'{coluna_origem}'."
            )

        except Exception as e:
            print(f"[ERRO] Falha ao gravar tabela '{tbl_ando}': {e}")

    except Exception as e:
        print(f"[ERRO CRÍTICO] Falha geral na função: {e}")

def ler_variavel_ambiente_spark(nome_variavel: str) -> str:
    valor = os.environ.get(nome_variavel)

    if valor is None or not valor.strip():
        raise ErroOperacional(
            codigo="CONFIG_VARIAVEL_AUSENTE",
            mensagem="Variavel de ambiente obrigatoria nao informada.",
            objeto=nome_variavel,
            acao="Configurar a variavel no ambiente da sessao Spark antes de executar a rotina.",
        )

    return valor.strip()


def registrar_erro_rotina(etapa, exc):
    erro_classificado = isinstance(exc, ErroPipeline)
    causa = _causa_raiz(exc)
    etapa_final = exc.etapa if erro_classificado and exc.etapa else etapa
    codigo = exc.codigo if erro_classificado else "ERRO_NAO_CLASSIFICADO"
    mensagem = exc.mensagem if erro_classificado else str(exc)
    objeto = exc.objeto if erro_classificado else None
    detalhes = exc.detalhes if erro_classificado else {}
    acao = exc.acao if erro_classificado else None

    detalhes_seguros = _detalhes_log_seguros(detalhes)
    linhas = [
        "[PIPELINE][ERRO_FATAL]",
        f"etapa={_texto_log_resumido(etapa_final or '')}",
        f"codigo={codigo}",
        f"tipo={type(exc).__name__}",
        f"objeto={_texto_log_resumido(objeto or '')}",
        f"mensagem={_texto_log_resumido(mensagem)}",
        f"detalhes={detalhes_seguros}",
        "causa_raiz="
        f"{type(causa).__name__}: {_texto_log_resumido(causa)}",
        f"acao={_texto_log_resumido(acao or '')}",
    ]

    logger_rotina.error("\n".join(linhas))
    logger_rotina.error("[PIPELINE][TRACEBACK]\n" + _redigir_segredos(traceback.format_exc()))


def _use_logs(default=True) -> bool:
    value = os.environ.get("USE_LOGS")

    if value is None:
        return default

    return str(value).strip().lower() == "true"


def _flush() -> None:
    try:
        sys.stdout.flush()
    except Exception:
        pass

    try:
        sys.stderr.flush()
    except Exception:
        pass


def _print_log(msg: str) -> None:
    print(msg)
    _flush()


class ScreenLogger:
    def __init__(self, name: str = "PIPELINE") -> None:
        self.name = name

    def step(self, suffix: str) -> "ScreenLogger":
        return ScreenLogger(name=f"{self.name}.{suffix}")

    def _prefix(self) -> str:
        return f"[{self.name}] "

    def info(self, msg: str) -> None:
        if not _use_logs(True):
            return

        _print_log(f"{self._prefix()}{msg}")

    def error(self, msg: str) -> None:
        if not _use_logs(True):
            return

        _print_log(f"{self._prefix()}{msg}")

    def obj(self, value, title: str = None) -> None:
        if not _use_logs(True):
            return

        if title:
            _print_log(f"{self._prefix()}{title}")

        _print_log(str(value))

    def df(self, df, n: int = 20, truncate=True, title: str = None) -> None:
        if not _use_logs(True):
            return

        if title:
            _print_log(f"{self._prefix()}{title}")

        df.show(n=n, truncate=truncate)
        _flush()

    def df_schema(self, df, title: str = None) -> None:
        if not _use_logs(True):
            return

        if title:
            _print_log(f"{self._prefix()}{title}")

        df.printSchema()
        _flush()


class NullLogger:
    def step(self, suffix: str):
        return self

    def info(self, msg: str) -> None:
        pass

    def error(self, msg: str) -> None:
        pass

    def obj(self, value, title: str = None) -> None:
        pass

    def df(self, df, n: int = 20, truncate=True, title: str = None) -> None:
        pass

    def df_schema(self, df, title: str = None) -> None:
        pass


def criar_logger_spark(nome: str = "PIPELINE"):
    if _use_logs(True):
        return ScreenLogger(nome)

    return NullLogger()


logger = criar_logger_spark("PIPELINE")
logger_rotina = logger


class ClientOracleSpark:
    DEFAULT_DRIVER = "oracle.jdbc.OracleDriver"

    def __init__(self, spark: SparkSession, env: Optional[Dict[str, str]] = None) -> None:
        self.spark = spark
        self.env = env or dict(os.environ)

        def env_required(key: str) -> str:
            value = self.env.get(key)

            if value is None or str(value).strip() == "":
                raise ErroOperacional(
                    codigo="CONFIG_VARIAVEL_AUSENTE",
                    mensagem="Variavel Oracle obrigatoria nao informada.",
                    objeto=key,
                    acao="Configurar a variavel Oracle no ambiente da sessao Spark.",
                )

            return str(value).strip()

        def env_optional(key: str) -> Optional[str]:
            value = self.env.get(key)

            if value is None or str(value).strip() == "":
                return None

            return str(value).strip()

        self.user = env_required("VDP_ORACLE_USER")
        self.password = env_required("VDP_ORACLE_PASSWORD")
        self.host_1 = env_required("VDP_ORACLE_HOST_1")
        self.host_2 = env_optional("VDP_ORACLE_HOST_2")
        self.port = env_optional("VDP_ORACLE_PORTA") or "1521"
        self.service_name = env_optional("VDP_ORACLE_SERVICE_NAME") or env_required("VDP_ORACLE_SERVICE")
        self.schema = (env_optional("VDP_ORACLE_SCHEMA") or self.user).upper()
        self.driver = env_optional("VDP_ORACLE_DRIVER") or self.DEFAULT_DRIVER
        self.jar_path = env_optional("VDP_ORACLE_JAR") or "/dados/shared/bin/ojdbc8.jar"

        if self.driver != self.DEFAULT_DRIVER:
            raise ErroOperacional(
                codigo="CONFIG_DRIVER_INVALIDO",
                mensagem="Driver Oracle configurado e invalido.",
                objeto="VDP_ORACLE_DRIVER",
                detalhes={"driver_configurado": self.driver},
                acao=f"Configurar o driver Oracle esperado: {self.DEFAULT_DRIVER}.",
            )

        if self.host_2:
            self.url = (
                "jdbc:oracle:thin:@(DESCRIPTION="
                "(LOAD_BALANCE=OFF)"
                "(FAILOVER=ON)"
                "(CONNECT_TIMEOUT=10)"
                "(TRANSPORT_CONNECT_TIMEOUT=3)"
                "(RETRY_COUNT=3)"
                "(ADDRESS_LIST="
                f"(ADDRESS=(PROTOCOL=TCP)(HOST={self.host_1})(PORT={self.port}))"
                f"(ADDRESS=(PROTOCOL=TCP)(HOST={self.host_2})(PORT={self.port}))"
                ")"
                f"(CONNECT_DATA=(SERVICE_NAME={self.service_name}))"
                ")"
            )
        else:
            self.url = f"jdbc:oracle:thin:@//{self.host_1}:{self.port}/{self.service_name}"

    def run_select(
        self,
        sql: str,
        fetchsize: Optional[int] = None,
    ) -> DataFrame:
        query = (sql or "").strip()

        if not query:
            raise ValueError("sql nao pode ser vazio")

        if query.endswith(";"):
            query = query[:-1].strip()

        reader = (
            self.spark.read
            .format("jdbc")
            .option("url", self.url)
            .option("driver", self.driver)
            .option("user", self.user)
            .option("password", self.password)
            .option("dbtable", f"({query}) T")
        )

        if fetchsize is not None:
            if int(fetchsize) <= 0:
                raise ValueError("fetchsize deve ser maior que zero")

            reader = reader.option("fetchsize", int(fetchsize))

        try:
            return reader.load()
        except ErroPipeline:
            raise
        except Exception as exc:
            raise ErroAcessoDados(
                codigo="ORACLE_LEITURA_FALHOU",
                mensagem="Falha ao executar leitura Oracle.",
                objeto=f"ORACLE:{self.schema}",
                detalhes={"operacao": "SELECT"},
                acao="Verificar conexao, permissao, disponibilidade e objeto consultado no Oracle.",
            ) from exc

    def selecionar_tabela(
            self,
            nome_tabela: str,
            owner: Optional[str] = None,
            fetchsize: Optional[int] = None,
            show: bool = False,
            truncate: bool = True,
            n: int = 20,
        ) -> DataFrame:
            owner_final = (owner or self.schema or "").strip().upper()
            tabela_final = (nome_tabela or "").strip().upper()

            if not owner_final:
                raise ValueError("owner/schema nao pode ser vazio")

            if not tabela_final:
                raise ValueError("nome_tabela nao pode ser vazio")

            if "." in tabela_final:
                raise ValueError(
                    "nome_tabela deve receber apenas o nome da tabela. "
                    "Informe owner/schema separadamente."
                )

            if int(n) <= 0:
                raise ValueError("n deve ser maior que zero")

            df = self.run_select(
                sql=f"SELECT * FROM {owner_final}.{tabela_final}",
                fetchsize=fetchsize,
            )

            if show:
                df.show(n=int(n), truncate=truncate)

            return df

    def execute(self, sql: str) -> None:
            command = (sql or "").strip()

            if not command:
                raise ValueError("sql nao pode ser vazio")

            if command.endswith(";"):
                command = command[:-1].strip()

            jvm = self.spark._jvm
            conn = None
            stmt = None

            def criar_conexao_driver_manager():
                try:
                    jvm.java.lang.Class.forName(self.driver)
                except Exception:
                    context_loader = (
                        jvm.java.lang.Thread
                        .currentThread()
                        .getContextClassLoader()
                    )

                    jvm.java.lang.Class.forName(
                        self.driver,
                        True,
                        context_loader
                    )

                return jvm.java.sql.DriverManager.getConnection(
                    self.url,
                    self.user,
                    self.password,
                )

            def criar_conexao_url_classloader():
                gateway = self.spark.sparkContext._gateway

                jar_file = jvm.java.io.File(self.jar_path)

                if not jar_file.exists():
                    raise ValueError(
                        f"Jar Oracle nao encontrado em: {self.jar_path}. "
                        "Informe VDP_ORACLE_JAR ou carregue o ojdbc8.jar na sessao Spark."
                    )

                jar_url = jar_file.toURI().toURL()

                urls = gateway.new_array(jvm.java.net.URL, 1)
                urls[0] = jar_url

                parent_loader = (
                    jvm.java.lang.Thread
                    .currentThread()
                    .getContextClassLoader()
                )

                loader = jvm.java.net.URLClassLoader(urls, parent_loader)

                driver_class = jvm.java.lang.Class.forName(
                    self.driver,
                    True,
                    loader
                )

                driver = driver_class.newInstance()

                props = jvm.java.util.Properties()
                props.setProperty("user", self.user)
                props.setProperty("password", self.password)

                return driver.connect(self.url, props)

            try:
                try:
                    conn = criar_conexao_driver_manager()
                except Exception:
                    conn = criar_conexao_url_classloader()

                conn.setAutoCommit(False)

                stmt = conn.createStatement()
                stmt.execute(command)

                conn.commit()

            except ErroPipeline:
                if conn is not None:
                    try:
                        conn.rollback()
                    except Exception:
                        pass

                raise

            except Exception as exc:
                if conn is not None:
                    try:
                        conn.rollback()
                    except Exception:
                        pass

                raise ErroAcessoDados(
                    codigo="ORACLE_COMANDO_FALHOU",
                    mensagem="Falha ao executar comando Oracle.",
                    objeto=f"ORACLE:{self.schema}",
                    detalhes={"operacao": "COMANDO_SEM_RETORNO"},
                    acao="Verificar conexao, permissao, disponibilidade e comando executado no Oracle.",
                ) from exc

            finally:
                if stmt is not None:
                    try:
                        stmt.close()
                    except Exception:
                        pass

                if conn is not None:
                    try:
                        conn.close()
                    except Exception:
                        pass

            return None

    def carregar_df(
            self,
            df: DataFrame,
            table_name: str,
            owner: Optional[str] = None,
            batchsize: int = 5000,
            num_partitions: int = 1,
        ) -> None:
            if df is None:
                raise ValueError("df nao pode ser None")

            if int(batchsize) <= 0:
                raise ValueError("batchsize deve ser maior que zero")

            if int(num_partitions) <= 0:
                raise ValueError("num_partitions deve ser maior que zero")

            table = (table_name or "").strip()

            if not table:
                raise ValueError("table_name nao pode ser vazio")

            if "." in table:
                raise ValueError(
                    "table_name deve receber apenas o nome da tabela. "
                    "Informe owner/schema separadamente."
                )

            final_owner = (owner or self.schema or "").strip()

            if not final_owner:
                raise ValueError("owner/schema nao pode ser vazio")

            full_table_name = f"{final_owner.upper()}.{table.upper()}"

            writer_df = df.coalesce(int(num_partitions))

            try:
                (
                    writer_df.write
                    .format("jdbc")
                    .mode("append")
                    .option("url", self.url)
                    .option("driver", self.driver)
                    .option("user", self.user)
                    .option("password", self.password)
                    .option("dbtable", full_table_name)
                    .option("batchsize", int(batchsize))
                    .save()
                )
            except ErroPipeline:
                raise
            except Exception as exc:
                raise ErroAcessoDados(
                    codigo="ORACLE_ESCRITA_FALHOU",
                    mensagem="Falha ao gravar DataFrame no Oracle.",
                    objeto=full_table_name,
                    detalhes={"operacao": "APPEND"},
                    acao="Verificar conexao, permissao, disponibilidade e limites fisicos da tabela Oracle.",
                ) from exc

            return None

    def limpar_tabela(
            self,
            table_name: str,
            owner: Optional[str] = None,
            use_truncate: bool = False,
            batchsize: int = 5000,
        ) -> None:
            if int(batchsize) <= 0:
                raise ValueError("batchsize deve ser maior que zero")
            
            table = (table_name or "").strip()

            if not table:
                raise ValueError("table_name nao pode ser vazio")

            if "." in table:
                raise ValueError(
                    "table_name deve receber apenas o nome da tabela. "
                    "Informe owner/schema separadamente."
                )

            final_owner = (owner or self.schema or "").strip()

            if not final_owner:
                raise ValueError("owner/schema nao pode ser vazio")

            full_table_name = f"{final_owner.upper()}.{table.upper()}"

            if use_truncate:
                self.execute(f"TRUNCATE TABLE {full_table_name}")
                return None

            lote = int(batchsize)

            while True:
                df_count = self.run_select(
                    f"SELECT COUNT(1) AS QTD FROM {full_table_name}"
                )

                qtd_restante = int(df_count.collect()[0]["QTD"])

                if qtd_restante == 0:
                    break

                self.execute(
                    f"DELETE FROM {full_table_name} WHERE ROWNUM <= {lote}"
                )

            return None

    def reload_dataframe(
            self,
            df: DataFrame,
            table_name: str,
            owner: Optional[str] = None,
            batchsize: int = 5000,
            num_partitions: int = 1,
            use_truncate: bool = False,
        ) -> None:
            if df is None:
                raise ValueError("df nao pode ser None")

            table = (table_name or "").strip()

            if not table:
                raise ValueError("table_name nao pode ser vazio")

            if "." in table:
                raise ValueError(
                    "table_name deve receber apenas o nome da tabela. "
                    "Informe owner/schema separadamente."
                )

            final_owner = (owner or self.schema or "").strip()

            if not final_owner:
                raise ValueError("owner/schema nao pode ser vazio")

            self.limpar_tabela(
                table_name=table,
                owner=final_owner,
                use_truncate=use_truncate,
                batchsize=batchsize,
            )

            self.carregar_df(
                df=df,
                table_name=table,
                owner=final_owner,
                batchsize=batchsize,
                num_partitions=num_partitions,
            )

            return None

def criar_cliente_oracle_spark(env: Optional[Dict[str, str]] = None) -> ClientOracleSpark:
    return ClientOracleSpark(spark=spark, env=env)


class ClientDb2Spark:
    DEFAULT_DRIVER = "com.ibm.db2.jcc.DB2Driver"

    def __init__(self, spark: SparkSession, env: Optional[Dict[str, str]] = None) -> None:
        self.spark = spark
        self.env = env or dict(os.environ)

        def env_required(key: str) -> str:
            value = self.env.get(key)

            if value is None or str(value).strip() == "":
                raise ErroOperacional(
                    codigo="CONFIG_VARIAVEL_AUSENTE",
                    mensagem="Variavel DB2 obrigatoria nao informada.",
                    objeto=key,
                    acao="Configurar a variavel DB2 no ambiente da sessao Spark.",
                )

            return str(value).strip()

        def env_optional(key: str) -> Optional[str]:
            value = self.env.get(key)

            if value is None or str(value).strip() == "":
                return None

            return str(value).strip()

        self.user = env_required("DB2_USER")
        self.password = env_required("DB2_PASSWORD")
        self.host = env_required("DB2_HOST")
        self.port = env_optional("DB2_PORTA") or env_optional("VDP_DB2_PORTA") or "50100"
        self.database = env_required("DB2_DATABASE")
        self.driver = env_optional("DB2_DRIVER") or self.DEFAULT_DRIVER

        if self.driver != self.DEFAULT_DRIVER:
            raise ErroOperacional(
                codigo="CONFIG_DRIVER_INVALIDO",
                mensagem="Driver DB2 configurado e invalido.",
                objeto="DB2_DRIVER",
                detalhes={"driver_configurado": self.driver},
                acao=f"Configurar o driver DB2 esperado: {self.DEFAULT_DRIVER}.",
            )

        self.url = f"jdbc:db2://{self.host}:{self.port}/{self.database}"

    def run_select(
        self,
        sql: str,
        fetchsize: Optional[int] = None,
        partition_column: Optional[str] = None,
        lower_bound: Optional[int] = None,
        upper_bound: Optional[int] = None,
        num_partitions: Optional[int] = None,
        query_timeout: Optional[int] = None,
    ) -> DataFrame:
        query = (sql or "").strip()

        if not query:
            raise ValueError("sql nao pode ser vazio")

        if query.endswith(";"):
            query = query[:-1].strip()

        reader = (
            self.spark.read
            .format("jdbc")
            .option("url", self.url)
            .option("driver", self.driver)
            .option("user", self.user)
            .option("password", self.password)
            .option("dbtable", f"({query}) T")
        )

        if fetchsize is not None:
            if int(fetchsize) <= 0:
                raise ValueError("fetchsize deve ser maior que zero")

            reader = reader.option("fetchsize", int(fetchsize))

        if query_timeout is not None:
            if int(query_timeout) <= 0:
                raise ValueError("query_timeout deve ser maior que zero")

            reader = reader.option("queryTimeout", int(query_timeout))

        parametros_particao = [
            partition_column,
            lower_bound,
            upper_bound,
            num_partitions,
        ]

        if any(valor is not None for valor in parametros_particao):
            if any(valor is None for valor in parametros_particao):
                raise ValueError(
                    "Para leitura particionada, informe partition_column, "
                    "lower_bound, upper_bound e num_partitions."
                )

            partition_column = (partition_column or "").strip()
            lower_bound_final = int(lower_bound)
            upper_bound_final = int(upper_bound)
            num_partitions_final = int(num_partitions)

            if not partition_column:
                raise ValueError("partition_column nao pode ser vazio")

            if lower_bound_final >= upper_bound_final:
                raise ValueError("lower_bound deve ser menor que upper_bound")

            if num_partitions_final <= 0:
                raise ValueError("num_partitions deve ser maior que zero")

            reader = (
                reader
                .option("partitionColumn", partition_column)
                .option("lowerBound", lower_bound_final)
                .option("upperBound", upper_bound_final)
                .option("numPartitions", num_partitions_final)
            )

        try:
            return reader.load()
        except ErroPipeline:
            raise
        except Exception as exc:
            raise ErroAcessoDados(
                codigo="DB2_LEITURA_FALHOU",
                mensagem="Falha ao executar leitura DB2.",
                objeto=f"DB2:{self.database}",
                detalhes={"operacao": "SELECT"},
                acao="Verificar conexao, permissao, disponibilidade e objeto consultado no DB2.",
            ) from exc

    def selecionar_tabela(
        self,
        schema: str,
        nome_tabela: str,
        fetchsize: Optional[int] = None,
        partition_column: Optional[str] = None,
        lower_bound: Optional[int] = None,
        upper_bound: Optional[int] = None,
        num_partitions: Optional[int] = None,
    ) -> DataFrame:
        schema_final = (schema or "").strip().upper()
        tabela_final = (nome_tabela or "").strip().upper()

        if not schema_final:
            raise ValueError("schema nao pode ser vazio")

        if not tabela_final:
            raise ValueError("nome_tabela nao pode ser vazio")

        if "." in tabela_final:
            raise ValueError(
                "nome_tabela deve receber apenas o nome da tabela. "
                "Informe o schema separadamente."
            )

        return self.run_select(
            sql=f"SELECT * FROM {schema_final}.{tabela_final}",
            fetchsize=fetchsize,
            partition_column=partition_column,
            lower_bound=lower_bound,
            upper_bound=upper_bound,
            num_partitions=num_partitions,
        )


def criar_cliente_db2_spark(env: Optional[Dict[str, str]] = None) -> ClientDb2Spark:
    return ClientDb2Spark(spark=spark, env=env)


def help_gerenciador_sessao_spark_remoto() -> None:
    print("""
============================================================
gerenciador_sessao_spark_remoto.ipynb
============================================================

Este notebook deve ser executado depois da criacao da sessao Spark.

Ele disponibiliza na sessao Spark:

- ler_variavel_ambiente_spark
- criar_logger_spark
- logger
- ClientOracleSpark
- criar_cliente_oracle_spark
- ClientDb2Spark
- criar_cliente_db2_spark

Leituras Oracle e DB2 usam spark.read.format("jdbc").options(...).load().
Escritas Oracle usam writer JDBC Spark.
Comandos Oracle sem retorno usam conexao JDBC direta.

============================================================
""")
